**Name: Rishi Phadale**  
**Roll No.: 23102C0070**  

# Assignment 2.1: Control Flow for Data Cleaning
**Topic:** Loops, Functions, and Error Handling in R

**Dataset:** UCI Heart Disease Dataset (Cleveland subset)

**Variable of interest:** `trestbps` (resting blood pressure)

## 0. Setup: install/load required packages

`microbenchmark` is used for precise timing comparisons in Task 3. If it isn't already installed on the Colab R runtime, this will install it (may take a minute).

In [1]:
packages_needed <- c("microbenchmark")
to_install <- packages_needed[!(packages_needed %in% installed.packages()[, "Package"])]
if (length(to_install) > 0) {
  install.packages(to_install, repos = "https://cloud.r-project.org")
}
library(microbenchmark)

set.seed(42)  # reproducibility

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



## 1. Load the UCI Heart Disease dataset

We attempt to download the real Cleveland Heart Disease dataset from the UCI repository.
If the download fails (e.g., no internet access in the sandbox), we fall back to a
synthetically generated dataset with the same column structure, so the rest of the
notebook always runs end-to-end.

In [2]:
col_names <- c("age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
               "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target")

load_uci_heart_data <- function() {
  url <- "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
  df <- read.csv(url, header = FALSE, na.strings = "?", col.names = col_names)
  df
}

simulate_heart_data <- function(n = 303) {
  data.frame(
    age      = round(rnorm(n, 54, 9)),
    sex      = sample(0:1, n, replace = TRUE),
    cp       = sample(0:3, n, replace = TRUE),
    trestbps = round(rnorm(n, 131, 17)),
    chol     = round(rnorm(n, 246, 51)),
    fbs      = sample(0:1, n, replace = TRUE),
    restecg  = sample(0:2, n, replace = TRUE),
    thalach  = round(rnorm(n, 149, 23)),
    exang    = sample(0:1, n, replace = TRUE),
    oldpeak  = round(abs(rnorm(n, 1, 1.2)), 1),
    slope    = sample(0:2, n, replace = TRUE),
    ca       = sample(0:3, n, replace = TRUE),
    thal     = sample(c(3, 6, 7), n, replace = TRUE),
    target   = sample(0:1, n, replace = TRUE)
  )
}

heart_data <- tryCatch({
  df <- load_uci_heart_data()
  message("Loaded real UCI Heart Disease dataset (", nrow(df), " rows).")
  df
}, error = function(e) {
  message("Could not download UCI dataset (", conditionMessage(e), "). Using simulated dataset instead.")
  simulate_heart_data()
})

str(heart_data)
head(heart_data)

Loaded real UCI Heart Disease dataset (303 rows).



'data.frame':	303 obs. of  14 variables:
 $ age     : num  63 67 67 37 41 56 62 57 63 53 ...
 $ sex     : num  1 1 1 1 0 1 0 0 1 1 ...
 $ cp      : num  1 4 4 3 2 2 4 4 4 4 ...
 $ trestbps: num  145 160 120 130 130 120 140 120 130 140 ...
 $ chol    : num  233 286 229 250 204 236 268 354 254 203 ...
 $ fbs     : num  1 0 0 0 0 0 0 0 0 1 ...
 $ restecg : num  2 2 2 0 2 0 2 0 2 2 ...
 $ thalach : num  150 108 129 187 172 178 160 163 147 155 ...
 $ exang   : num  0 1 1 0 0 0 0 1 0 1 ...
 $ oldpeak : num  2.3 1.5 2.6 3.5 1.4 0.8 3.6 0.6 1.4 3.1 ...
 $ slope   : num  3 2 2 3 1 1 3 1 2 3 ...
 $ ca      : num  0 3 2 0 0 0 2 0 1 0 ...
 $ thal    : num  6 3 7 3 3 3 3 3 7 7 ...
 $ target  : int  0 2 1 0 0 0 3 0 2 1 ...


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,63,1,1,145,233,1,2,150,0,2.3,3,0,6,0
2,67,1,4,160,286,0,2,108,1,1.5,2,3,3,2
3,67,1,4,120,229,0,2,129,1,2.6,2,2,7,1
4,37,1,3,130,250,0,0,187,0,3.5,3,0,3,0
5,41,0,2,130,204,0,2,172,0,1.4,1,0,3,0
6,56,1,2,120,236,0,0,178,0,0.8,1,0,3,0


## 2. Simulate realistic data-entry problems in `trestbps`

We deliberately corrupt a handful of `trestbps` values to mimic real-world clinical
data-entry errors:
- A few **negative** BP values (impossible measurements)
- A few **missing (NA)** values
- A few **extreme** values greater than 300 mmHg

In [3]:
n <- nrow(heart_data)
heart_data$trestbps <- as.numeric(heart_data$trestbps)

# Choose disjoint sets of row indices to corrupt
idx_all      <- sample(seq_len(n), size = min(15, n))
idx_negative <- idx_all[1:5]
idx_missing  <- idx_all[6:10]
idx_extreme  <- idx_all[11:15]

heart_data$trestbps[idx_negative] <- -abs(heart_data$trestbps[idx_negative])   # negative BP
heart_data$trestbps[idx_missing]  <- NA                                        # missing BP
heart_data$trestbps[idx_extreme]  <- round(runif(5, 305, 400))                 # extreme BP (>300)

cat("Injected problems into trestbps:\n")
cat(" - Negative values at rows:", idx_negative, "\n")
cat(" - Missing values at rows:", idx_missing, "\n")
cat(" - Extreme (>300) values at rows:", idx_extreme, "\n\n")

summary(heart_data$trestbps)

Injected problems into trestbps:
 - Negative values at rows: 49 153 74 228 146 
 - Missing values at rows: 122 303 128 24 89 
 - Extreme (>300) values at rows: 165 110 20 294 283 



   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.     NAs 
 -152.0   120.0   130.0   131.4   140.0   391.0       5 

## Task 1: BP-cleaning function using if-else

`clean_bp_value()` applies the following rules to a **single** BP reading:
- If negative -> convert to `NA`
- If greater than 250 -> cap at 250
- Otherwise -> keep unchanged (including existing `NA`s, which pass through)

`clean_bp_vector()` applies this rule to an entire vector by looping over each
element and calling `clean_bp_value()` — this is the loop-based implementation
used later in Task 3 as well.

In [4]:
# --- Single-value cleaning function using if-else ---
clean_bp_value <- function(bp) {
  if (is.na(bp)) {
    return(NA)
  } else if (bp < 0) {
    return(NA)          # invalid negative reading -> NA
  } else if (bp > 250) {
    return(250)          # cap extreme outlier
  } else {
    return(bp)            # valid value, keep as-is
  }
}

# --- Loop-based application over a vector (used again in Task 3) ---
clean_bp_vector_loop <- function(bp_vector) {
  cleaned <- numeric(length(bp_vector))
  for (i in seq_along(bp_vector)) {
    cleaned[i] <- clean_bp_value(bp_vector[i])
  }
  cleaned
}

# Quick test of the function on a few sample values
test_values <- c(-10, 0, 120, 251, 300, NA, 250)
data.frame(original = test_values, cleaned = sapply(test_values, clean_bp_value))

original,cleaned
<dbl>,<dbl>
-10,NA
0,0
120,120
251,250
300,250
NA,NA
250,250


## Task 2: Error handling with `tryCatch()`

Two robust functions are defined below:

1. `safe_mean_bp()` — safely computes the mean of a BP vector even when missing
   values, non-numeric input, or an empty vector are present.
2. `safe_ratio()` — safely computes `chol / trestbps`, handling the case where the
   denominator is zero, `NA`, negative, or otherwise invalid, and returns an
   informative message instead of crashing.

In [5]:
# --- Safe mean calculation ---
safe_mean_bp <- function(bp_vector) {
  tryCatch({
    if (!is.numeric(bp_vector)) {
      stop("Input is not numeric.")
    }
    if (all(is.na(bp_vector))) {
      stop("All BP values are missing; mean cannot be computed.")
    }
    result <- mean(bp_vector, na.rm = TRUE)
    message("Mean BP successfully computed: ", round(result, 2))
    return(result)
  }, warning = function(w) {
    message("Warning while computing mean BP: ", conditionMessage(w))
    return(NA)
  }, error = function(e) {
    message("Error while computing mean BP: ", conditionMessage(e))
    return(NA)
  })
}

# --- Safe ratio calculation: chol / trestbps ---
safe_ratio <- function(chol, trestbps) {
  tryCatch({
    if (is.na(chol) || is.na(trestbps)) {
      stop("Cannot compute ratio: chol or trestbps is NA.")
    }
    if (!is.numeric(chol) || !is.numeric(trestbps)) {
      stop("Cannot compute ratio: inputs must be numeric.")
    }
    if (trestbps == 0) {
      stop("Cannot compute ratio: division by zero (trestbps = 0).")
    }
    if (trestbps < 0) {
      warning("trestbps is negative; ratio may not be clinically meaningful.")
    }
    chol / trestbps
  }, warning = function(w) {
    message("Warning in safe_ratio(): ", conditionMessage(w))
    return(chol / trestbps)
  }, error = function(e) {
    message("Error in safe_ratio(): ", conditionMessage(e))
    return(NA)
  })
}

# --- Demonstrations ---
cat("Demo: safe_mean_bp() on raw (uncleaned) trestbps with NAs present\n")
invisible(safe_mean_bp(heart_data$trestbps))

cat("\nDemo: safe_ratio() edge cases\n")
cat("chol=200, trestbps=100  ->", safe_ratio(200, 100), "\n")
cat("chol=200, trestbps=0    ->", safe_ratio(200, 0), "\n")
cat("chol=200, trestbps=NA   ->", safe_ratio(200, NA), "\n")
cat("chol=200, trestbps=-10  ->", safe_ratio(200, -10), "\n")

Demo: safe_mean_bp() on raw (uncleaned) trestbps with NAs present


Mean BP successfully computed: 131.42




Demo: safe_ratio() edge cases
chol=200, trestbps=100  -> 2 


Error in safe_ratio(): Cannot compute ratio: division by zero (trestbps = 0).



chol=200, trestbps=0    -> NA 


Error in safe_ratio(): Cannot compute ratio: chol or trestbps is NA.



chol=200, trestbps=NA   -> NA 


Warning in safe_ratio(): trestbps is negative; ratio may not be clinically meaningful.



chol=200, trestbps=-10  -> -20 


Applying `safe_ratio()` row-wise across the dataset with `mapply()` (itself wrapped
so a single bad row can't halt the whole computation):

In [6]:
heart_data$chol_bp_ratio <- mapply(safe_ratio, heart_data$chol, heart_data$trestbps)
summary(heart_data$chol_bp_ratio)

Error in safe_ratio(): Cannot compute ratio: chol or trestbps is NA.

Warning in safe_ratio(): trestbps is negative; ratio may not be clinically meaningful.

Warning in safe_ratio(): trestbps is negative; ratio may not be clinically meaningful.

Error in safe_ratio(): Cannot compute ratio: chol or trestbps is NA.

Error in safe_ratio(): Cannot compute ratio: chol or trestbps is NA.

Error in safe_ratio(): Cannot compute ratio: chol or trestbps is NA.

Warning in safe_ratio(): trestbps is negative; ratio may not be clinically meaningful.

Warning in safe_ratio(): trestbps is negative; ratio may not be clinically meaningful.

Warning in safe_ratio(): trestbps is negative; ratio may not be clinically meaningful.

Error in safe_ratio(): Cannot compute ratio: chol or trestbps is NA.



   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.     NAs 
 -4.904   1.553   1.823   1.784   2.134   3.118       5 

## Task 3: Loop-based vs vectorized cleaning — performance comparison

We compare two ways of identifying/cleaning invalid `trestbps` values:

- **Loop-based**: iterate element-by-element with a `for` loop and if-else logic
  (`clean_bp_vector_loop()`, already defined in Task 1).
- **Vectorized**: use R's vectorized `ifelse()` to apply the same rules to the
  whole vector at once, with no explicit loop.

Timing is measured with both `system.time()` (single run) and `microbenchmark()`
(many repeated runs, for a more reliable comparison).

In [7]:
# --- Vectorized cleaning function ---
clean_bp_vector_vectorized <- function(bp_vector) {
  ifelse(is.na(bp_vector), NA,
         ifelse(bp_vector < 0, NA,
                ifelse(bp_vector > 250, 250, bp_vector)))
}

# To make the timing difference clearly visible, replicate trestbps into a larger vector
big_bp <- rep(heart_data$trestbps, times = 2000)   # simulate a much larger dataset
cat("Benchmark vector length:", length(big_bp), "\n\n")

cat("---- system.time(): loop-based cleaning ----\n")
time_loop <- system.time({
  cleaned_loop <- clean_bp_vector_loop(big_bp)
})
print(time_loop)

cat("\n---- system.time(): vectorized cleaning ----\n")
time_vectorized <- system.time({
  cleaned_vectorized <- clean_bp_vector_vectorized(big_bp)
})
print(time_vectorized)

# Sanity check: both approaches must produce identical results
cat("\nResults identical between loop and vectorized approach:",
    identical(round(cleaned_loop, 6), round(cleaned_vectorized, 6)), "\n")

Benchmark vector length: 606000 

---- system.time(): loop-based cleaning ----
   user  system elapsed 
   0.33    0.00    0.33 

---- system.time(): vectorized cleaning ----
   user  system elapsed 
  0.034   0.006   0.040 

Results identical between loop and vectorized approach: TRUE 


In [8]:
cat("---- microbenchmark(): repeated timing comparison ----\n")
bench_results <- microbenchmark(
  loop_based = clean_bp_vector_loop(big_bp),
  vectorized = clean_bp_vector_vectorized(big_bp),
  times = 20
)
print(bench_results)

---- microbenchmark(): repeated timing comparison ----
Unit: milliseconds
       expr       min        lq      mean    median        uq      max neval
 loop_based 316.71625 323.02752 359.34950 328.48225 338.32717 597.2070    20
 vectorized  28.35605  33.21145  65.81977  65.66008  94.99414 123.7562    20


**Interpretation:** `microbenchmark` reports timing in microseconds/milliseconds
across 20 repetitions of each approach. In almost every case, the vectorized
version is substantially faster than the explicit `for` loop, because R's
vectorized operations are implemented in optimized, compiled (C-level) code and
avoid the per-iteration overhead of the R interpreter.

## Apply the cleaning function to the actual dataset

We now use the vectorized cleaner (validated above to be equivalent to the loop
version) to produce the final cleaned `trestbps` column.

In [9]:
heart_data$trestbps_cleaned <- clean_bp_vector_vectorized(heart_data$trestbps)

cat("Before vs after cleaning (rows that were originally corrupted):\n")
print(data.frame(
  row = idx_all,
  original = heart_data$trestbps[idx_all],
  cleaned  = heart_data$trestbps_cleaned[idx_all]
))

Before vs after cleaning (rows that were originally corrupted):
   row original cleaned
1   49     -140      NA
2  153     -115      NA
3   74     -110      NA
4  228     -152      NA
5  146     -108      NA
6  122       NA      NA
7  303       NA      NA
8  128       NA      NA
9   24       NA      NA
10  89       NA      NA
11 165      354     250
12 110      342     250
13  20      391     250
14 294      347     250
15 283      384     250


## Task 4: Validate the cleaned data

We verify that:
- The number of missing BP values matches expectations (originally-missing rows
  + newly-nullified negative rows)
- Min/max/mean/median are all within a plausible clinical range
- No negative or >250 values remain

In [10]:
bp_clean <- heart_data$trestbps_cleaned

n_missing <- sum(is.na(bp_clean))
bp_min    <- min(bp_clean, na.rm = TRUE)
bp_max    <- max(bp_clean, na.rm = TRUE)
bp_mean   <- mean(bp_clean, na.rm = TRUE)
bp_median <- median(bp_clean, na.rm = TRUE)

any_negative <- any(bp_clean < 0, na.rm = TRUE)
any_over_250 <- any(bp_clean > 250, na.rm = TRUE)

cat("---- Validation Summary: cleaned trestbps ----\n")
cat("Missing (NA) values     :", n_missing, "\n")
cat("Minimum                 :", bp_min, "\n")
cat("Maximum                 :", bp_max, "\n")
cat("Mean                    :", round(bp_mean, 2), "\n")
cat("Median                  :", bp_median, "\n")
cat("Any negative values left?:", any_negative, "\n")
cat("Any values > 250 left?   :", any_over_250, "\n")

stopifnot(!any_negative, !any_over_250)
cat("\nValidation PASSED: no negative or >250 values remain in cleaned data.\n")

---- Validation Summary: cleaned trestbps ----
Missing (NA) values     : 10 
Minimum                 : 94 
Maximum                 : 250 
Mean                    : 133.85 
Median                  : 130 
Any negative values left?: FALSE 
Any values > 250 left?   : FALSE 

Validation PASSED: no negative or >250 values remain in cleaned data.


## Save the cleaned dataset

In [11]:
output_path <- "cleaned_heart_data.csv"
write.csv(heart_data, output_path, row.names = FALSE)
cat("Saved cleaned dataset to:", normalizePath(output_path), "\n")

head(heart_data)

Saved cleaned dataset to: /content/cleaned_heart_data.csv 


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target,chol_bp_ratio,trestbps_cleaned
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>
1,63,1,1,145,233,1,2,150,0,2.3,3,0,6,0,1.606897,145
2,67,1,4,160,286,0,2,108,1,1.5,2,3,3,2,1.787500,160
3,67,1,4,120,229,0,2,129,1,2.6,2,2,7,1,1.908333,120
4,37,1,3,130,250,0,0,187,0,3.5,3,0,3,0,1.923077,130
5,41,0,2,130,204,0,2,172,0,1.4,1,0,3,0,1.569231,130
6,56,1,2,120,236,0,0,178,0,0.8,1,0,3,0,1.966667,120
